# Obtención del conjunto de datos de entrenamiento

En este notebook consultamos los datos de la base original (arquitectura inicial) del sistema para obtener la información de entrenamiento.

## Librerías
Cargamos las librerías y las variables de entorno para acceder al servidor de base de datos.

In [54]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import numpy as np

load_dotenv()

SERVER = os.getenv("MSSQL_SERVER")
DATABASE = os.getenv("MSSQL_DATABASE")
USERNAME = os.getenv("MSSQL_USER")
PASSWORD = os.getenv("MSSQL_PASSWORD")
DRIVER = os.getenv("MSSQL_DRIVER", "ODBC Driver 18 for SQL Server")
TRUST_SERVER_CERT = os.getenv("MSSQL_TRUST_SERVER_CERT", "yes")

## Obtención de los datos

Filtramos los datos de la tabla a partir de la fecha de instalación y finalización de pruebas del sistema en su arquitectura inicial hasta la ocurrencia de un salto en los datos capturados debido a fallas externas.

In [55]:
query = """
SELECT [FechaMedicion]
      ,[Fase]
      ,[Voltaje]
      ,[Corriente]
      ,[Potencia]
      ,[FactorPotencia]
      ,[Frecuencia]
      ,[EnergiaActiva]
  FROM [IoTData].[dbo].[Vw_IndicadoresEnergia]
WHERE FechaMedicion BETWEEN '2026-02-15 13:10:00' AND '2026-03-14 14:10:00'
ORDER BY FechaMedicion DESC
"""

odbc_str = (
    f"DRIVER={{{DRIVER}}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"UID={USERNAME};"
    f"PWD={PASSWORD};"
    f"TrustServerCertificate={TRUST_SERVER_CERT};"
)


connect_url = "mssql+pyodbc:///?odbc_connect=" + urllib.parse.quote_plus(odbc_str)
engine = create_engine(connect_url)

df = pd.read_sql(query, engine)

Verificamos el tipo de datos y el conteo

In [56]:
display(df.shape)
display(df.dtypes)
display(df.head())

(75318, 8)

FechaMedicion     datetime64[ns]
Fase                       int64
Voltaje                  float64
Corriente                float64
Potencia                 float64
FactorPotencia           float64
Frecuencia               float64
EnergiaActiva            float64
dtype: object

,FechaMedicion,Fase,Voltaje,Corriente,Potencia,FactorPotencia,Frecuencia,EnergiaActiva
0,2026-03-14 14:10:00,1,118.199997,1.814,181.600006,0.85,59.900002,230.800003
1,2026-03-14 14:10:00,2,114.699997,0.347,21.500000,0.54,60.000000,487.157013
2,2026-03-14 14:09:00,2,113.599998,0.342,20.799999,0.54,60.000000,487.157013
3,2026-03-14 14:09:00,1,118.099998,1.834,184.000000,0.85,60.000000,230.796997
4,2026-03-14 14:08:00,1,117.900002,3.308,335.399994,0.86,59.900002,230.792007


Ajustamos el formato de los decimales para redondear a 3 cifras

In [57]:
columnas_numericas = ["Voltaje", "Corriente", "Potencia", "FactorPotencia", "Frecuencia", "EnergiaActiva"]
df[columnas_numericas] = np.round(df[columnas_numericas], 3)

Agrupamos los datos para transformar la resolución por minuto a mediciones en intervalos de 10 minutos

In [58]:
df["FechaMedicion"] = pd.to_datetime(df["FechaMedicion"])

df_10min = (
    df.sort_values("FechaMedicion")
    .groupby(["Fase", pd.Grouper(key="FechaMedicion", freq="10min")])
    .agg(
        {
            "Voltaje": "mean",
            "Corriente": "mean",
            "Potencia": "mean",
            "FactorPotencia": "mean",
            "Frecuencia": "mean",
            "EnergiaActiva": "max",
        }
    )
    .reset_index()
)

print(f"Filas en df original: {len(df)}")
print(f"Filas en df_10min: {len(df_10min)}")
df_10min.head()

Filas en df original: 75318
Filas en df_10min: 7790


,Fase,FechaMedicion,Voltaje,Corriente,Potencia,FactorPotencia,Frecuencia,EnergiaActiva
0,1,2026-02-15 13:10:00,119.8100,3.859700,405.8900,0.87700,59.9700,0.159
1,1,2026-02-15 13:20:00,120.0375,3.535625,369.5125,0.87375,59.9625,0.222
2,1,2026-02-15 13:30:00,118.8600,4.576500,499.5500,0.90900,59.9700,0.297
3,1,2026-02-15 13:40:00,118.3300,8.892900,1003.4900,0.93400,59.9500,0.452
4,1,2026-02-15 13:50:00,119.2300,6.094000,696.6800,0.93700,59.9600,0.585


Validamos si existen datos cero o nulos

In [59]:
# Función para validar si en el dataframe hay valores nulos o ceros
def validar_datos(df):
    for column in df.columns:
        if df[column].isnull().any():
            print(f"Columna '{column}' contiene valores nulos.")
        if (df[column] == 0).any():
            print(f"Columna '{column}' contiene valores cero.")
    print("Validación de datos completada.")


validar_datos(df_10min)

Validación de datos completada.


Creamos una función para validar si existen saltos en los intervalos de los datos a utilizar

In [60]:
def validar_huecos_tiempo(
    df,
    columna_fecha="FechaMedicion",
    frecuencia="10min",
    columnas_grupo=[],
    mostrar_ejemplos=10,
):
    """
    Valida si existen huecos temporales en una serie de tiempo.

    Ejemplo de hueco: existe 13:10 y 13:30 pero falta 13:20.
    """
    data = df.copy()
    data[columna_fecha] = pd.to_datetime(data[columna_fecha], errors="coerce")
    data = data.dropna(subset=[columna_fecha]).sort_values(columna_fecha)

    if data.empty:
        print("No hay datos válidos para evaluar huecos de tiempo.")
        return pd.DataFrame()

    def _huecos_en_serie(serie_fechas):
        fechas_unicas = serie_fechas.drop_duplicates().sort_values()
        if fechas_unicas.empty:
            return []

        esperado = pd.date_range(
            start=fechas_unicas.min(), end=fechas_unicas.max(), freq=frecuencia
        )
        faltantes = esperado.difference(fechas_unicas)
        return list(faltantes)

    registros_huecos = []

    if columnas_grupo:
        for claves, grupo in data.groupby(columnas_grupo):
            claves = claves if isinstance(claves, tuple) else (claves,)
            faltantes = _huecos_en_serie(grupo[columna_fecha])
            for ts in faltantes:
                fila = {col: val for col, val in zip(columnas_grupo, claves)}
                fila["timestamp_faltante"] = ts
                registros_huecos.append(fila)
    else:
        faltantes = _huecos_en_serie(data[columna_fecha])
        registros_huecos = [{"timestamp_faltante": ts} for ts in faltantes]

    df_huecos = pd.DataFrame(registros_huecos)

    if df_huecos.empty:
        print(f"Sin huecos detectados para frecuencia '{frecuencia}'.")
    else:
        print(f"Se detectaron {len(df_huecos)} huecos para frecuencia '{frecuencia}'.")
        display(df_huecos.head(mostrar_ejemplos))

    return df_huecos


huecos_10min = validar_huecos_tiempo(
    df_10min, frecuencia="10min", columnas_grupo=["Fase"]
)

Sin huecos detectados para frecuencia '10min'.


## Guardamos los datos de entrenamiento

In [61]:
output_dir = Path("..") / "data" / "conjuntos"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "datos_entrenamiento.csv"

df_10min[columnas_numericas] = df_10min[columnas_numericas].round(3)

df_10min.to_csv(output_file, index=False)

# Datos de validación

Para validar los modelos emplearemos el siguiente tramo de información disponible

In [ ]:
consulta_validacion = """
SELECT [FechaMedicion]
      ,[Fase]
      ,[Voltaje]
      ,[Corriente]
      ,[Potencia]
      ,[FactorPotencia]
      ,[Frecuencia]
      ,[EnergiaActiva]
  FROM [IoTData].[dbo].[Vw_IndicadoresEnergia]
 WHERE FechaMedicion >= '2026-03-16 22:00:00'
   AND FechaMedicion <= '2026-03-29 22:00:00'
 ORDER BY FechaMedicion DESC
"""

df_prueba = pd.read_sql(consulta_validacion, engine)

print(f"Filas traídas desde SQL: {len(df_prueba)}")
print("Tipos de datos:")
print(df_prueba.dtypes)
display(df_prueba.head())

Filas traídas desde SQL (últimos 5 días): 34388
Tipos de datos:
FechaMedicion     datetime64[ns]
Fase                       int64
Voltaje                  float64
Corriente                float64
Potencia                 float64
FactorPotencia           float64
Frecuencia               float64
EnergiaActiva            float64
dtype: object


,FechaMedicion,Fase,Voltaje,Corriente,Potencia,FactorPotencia,Frecuencia,EnergiaActiva
0,2026-03-29 18:18:00,1,120.900002,3.806,399.000000,0.87,60.000000,375.635010
1,2026-03-29 18:18:00,2,114.900002,2.294,224.800003,0.85,60.000000,524.981018
2,2026-03-29 18:17:00,1,121.900002,3.949,410.000000,0.85,59.900002,375.628998
3,2026-03-29 18:17:00,2,114.400002,2.345,229.800003,0.86,59.900002,524.976990
4,2026-03-29 18:16:00,2,116.199997,2.256,223.899994,0.85,60.000000,524.973999


Agrupamos los datos

In [63]:
df_prueba["FechaMedicion"] = pd.to_datetime(df_prueba["FechaMedicion"])

df_prueba_10min = (
    df_prueba.sort_values("FechaMedicion")
    .groupby(["Fase", pd.Grouper(key="FechaMedicion", freq="10min")])
    .agg(
        {
            "Voltaje": "mean",
            "Corriente": "mean",
            "Potencia": "mean",
            "FactorPotencia": "mean",
            "Frecuencia": "mean",
            "EnergiaActiva": "max",
        }
    )
    .reset_index()
)

print(f"Filas en df original: {len(df_prueba)}")
print(f"Filas en df_prueba_10min: {len(df_prueba_10min)}")
df_prueba_10min.head()

Filas en df original: 34388
Filas en df_prueba_10min: 3700


,Fase,FechaMedicion,Voltaje,Corriente,Potencia,FactorPotencia,Frecuencia,EnergiaActiva
0,1,2026-03-16 22:00:00,116.755557,2.718778,283.077779,0.891111,59.977778,251.492004
1,1,2026-03-16 22:10:00,118.019999,3.163500,339.109998,0.898000,59.960001,251.550003
2,1,2026-03-16 22:20:00,114.600001,3.667444,384.466665,0.898889,59.977778,251.610001
3,1,2026-03-16 22:30:00,117.580000,4.451400,490.350000,0.907000,59.940001,251.699997
4,1,2026-03-16 22:40:00,117.000001,2.314778,227.522224,0.840000,59.944446,251.738007


Validamos los datos

In [64]:
validar_datos(df_prueba_10min)
display(
    validar_huecos_tiempo(df_prueba_10min, frecuencia="10min", columnas_grupo=["Fase"])
)

Validación de datos completada.
Sin huecos detectados para frecuencia '10min'.


""


Guardamos los datos de validación

In [65]:
output_file = output_dir / "datos_prueba.csv"

df_prueba_10min[columnas_numericas] = df_prueba_10min[columnas_numericas].round(3)

df_10min.to_csv(output_file, index=False)